# Jobs, Workers, and Development: CPS ASEC analysis

Step 1: mount Google Drive, load the IPUMS CPS ASEC file, and take a first look at it.

In [ ]:
# Import everything the notebook needs
import pandas as pd  # tables and data loading
import numpy as np  # weighted averages
import matplotlib.pyplot as plt  # charts (used in later steps)
from google.colab import drive  # lets Colab reach Google Drive

In [ ]:
# Mount Google Drive (Colab will ask you to allow access)
drive.mount('/content/drive')  # connect Drive at /content/drive

In [ ]:
# Load the data from Google Drive: /content/drive/My Drive/ghdp_7031_data.csv
file_path = '/content/drive/My Drive/ghdp_7031_data.csv'  # where the file lives in Drive
df_asec = pd.read_csv(file_path)  # read the CSV into a table

In [ ]:
# Show how big the table is: (number of rows, number of columns)
print(df_asec.shape)  # print rows and columns

In [ ]:
# Show the column names exactly as they are in the file
print(list(df_asec.columns))  # print every column name

In [ ]:
# IPUMS saves names in capitals (e.g. ASECWT); make them lowercase so they match our rules
df_asec.columns = df_asec.columns.str.lower()  # lowercase every column name

In [ ]:
# Summary table: weighted mean (using asecwt), plus the smallest and largest value
weight_col = 'asecwt'  # person-level survey weight
numeric_cols = df_asec.select_dtypes(include='number').columns.drop(weight_col)  # number columns, not the weight
summary_rows = []  # will hold one row of results per column
for col in numeric_cols:  # go through each number column
    has_value = df_asec[col].notna()  # True where the column is not missing
    values = df_asec.loc[has_value, col]  # the column's non-missing values
    weights = df_asec.loc[has_value, weight_col]  # the weights for those same people
    weighted_mean = np.average(values, weights=weights)  # average that counts each person by their weight
    summary_rows.append([col, weighted_mean, values.min(), values.max()])  # save this column's results
summary_table = pd.DataFrame(summary_rows, columns=['Variable', 'Weighted mean', 'Min', 'Max'])  # build the table
print(summary_table.to_string(index=False, float_format='{:,.1f}'.format))  # print with 1 decimal

In [ ]:
# Poverty rate (%) by year and work status last year, weighted with asecwt
assert set(df_asec['poor'].dropna().unique()) <= {0, 1}  # stop with an error if poor is not coded 0/1
df_poor = df_asec.dropna(subset=['poor']).copy()  # keep only people whose poverty status is known
df_poor['fulltime_label'] = df_poor['fulltime_label'].fillna('Did not work last year')  # missing work status means no work
df_poor['poor_x_wt'] = df_poor['poor'] * df_poor['asecwt']  # weight of each person who is poor (0 if not poor)
poor_wt = df_poor.groupby(['year', 'fulltime_label'])['poor_x_wt'].sum().unstack()  # weighted poor people: year by work status
all_wt = df_poor.groupby(['year', 'fulltime_label'])['asecwt'].sum().unstack()  # weighted all people: year by work status
poverty_by_year_work = 100 * poor_wt / all_wt  # poverty rate in percent for each cell
n_by_year_work = df_poor.groupby(['year', 'fulltime_label']).size().unstack().fillna(0).astype(int)  # unweighted respondents per cell
print('Table 1. Poverty rate (%), weighted with asecwt: rows = year, columns = work status last year')  # title for the table
print(poverty_by_year_work.to_string(float_format='{:.1f}'.format))  # print the rates with 1 decimal
print()  # blank line between the two tables
print('Unweighted number of respondents in each cell')  # title for the counts
print(n_by_year_work.to_string())  # print the respondent counts
print()  # blank line before the explanation
first_year = poverty_by_year_work.index[0]  # first year in the table
first_group = poverty_by_year_work.columns[0]  # first work-status group in the table
first_rate = poverty_by_year_work.loc[first_year, first_group]  # the rate in that cell
print(f'How to read one cell: in {first_year}, among people in the group "{first_group}", '  # start of the plain-English example
      f'{first_rate:.1f}% were below the poverty line, counting each person by their survey weight '  # say what the number is
      f'(so the number stands for the whole population, not just the people surveyed).')  # explain the weighting


In [ ]:
# Line chart of the poverty rate by year, one line per work status; saved to Google Drive
table_for_chart = poverty_by_year_work.round(1)  # the exact table the chart will show, rounded to 1 decimal
print('Table 1. Poverty rate (%), weighted with asecwt (the table shown in Figure 1)')  # label the table before the chart
print(table_for_chart.to_string(float_format='{:.1f}'.format))  # print the table with 1 decimal
last_year = table_for_chart.index[-1]  # latest year in the table
last_row = table_for_chart.loc[last_year].dropna()  # latest-year rates, skipping empty groups
high_group = last_row.idxmax()  # work status with the highest poverty rate in the latest year
low_group = last_row.idxmin()  # work status with the lowest poverty rate in the latest year
chart_title = (f'In {last_year}, the poverty rate was {last_row[high_group]:.1f}% among "{high_group}"\n'  # first half of the one-sentence title
               f'and {last_row[low_group]:.1f}% among "{low_group}"')  # second half of the title
fig, ax = plt.subplots(figsize=(10, 6))  # make an empty chart, 10 by 6 inches
for work_status in table_for_chart.columns:  # go through each work status
    ax.plot(table_for_chart.index, table_for_chart[work_status], marker='o', label=work_status)  # draw one line with dots
ax.set_xticks(table_for_chart.index)  # put an x tick on every year
ax.set_ylim(bottom=0)  # make the y-axis start at 0
ax.set_xlabel('Year')  # x-axis label
ax.set_ylabel('Poverty rate (% of people)')  # y-axis label with units
fig.suptitle('Figure 1', x=0.01, ha='left', fontsize=12, fontweight='bold')  # label the chart as Figure 1
ax.set_title(chart_title, fontsize=12)  # title that states the finding
ax.legend(title='Work status last year')  # legend naming each line
ax.grid(axis='y', alpha=0.3)  # light horizontal grid lines
fig.text(0.01, 0.01, 'Source: CPS ASEC 2020-2025 via IPUMS, weighted with asecwt', fontsize=9)  # source note at the bottom
fig.subplots_adjust(bottom=0.12)  # leave room for the source note
save_path = '/content/drive/My Drive/fig1_poverty_by_work.png'  # where to save the chart in Google Drive
fig.savefig(save_path, dpi=200)  # save the chart as a PNG
plt.show()  # show the chart in the notebook
print('Saved to', save_path)  # confirm where the file went


## Write-up (50-100 words)

Question: This is my question

Finding: This is my finding